# Results for model: abacusai/dracarys-llama-3.1-70b-instruct

In [ ]:
import polars as pl
import xgboost as xgb
from sklearn.model_selection import train_test_split

# Load the data
df = pl.read_parquet('./jane_street_data/train.parquet')

# Feature Engineering: Calculate a global TOP_QUANTILE (top 15%) of 'feature_00' 
# relative to 'responder_6' across rolling batches of 'date_id'
df = df.with_columns([
    pl.col('feature_00').over('date_id').quantile(0.85).alias('TOP_QUANTILE')
])

# Prepare the data for training
X = df.drop(['responder_6', 'date_id'])
y = df['responder_6']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train an XGBoost Regressor on the target 'responder_6'
xgb_model = xgb.XGBRegressor()
xgb_model.fit(X_train, y_train)